# 07 · Learning curves

**What this notebook decides.** How much task-specific supervision is actually
required — and whether collecting more data is worth it.

Three readings matter, in order:

1. **Where does the curve cross the unadapted baseline?** Below that point,
   fine-tuning is actively harmful.
2. **Where does it flatten?** Past that point, more data buys little and the
   bottleneck is elsewhere.
3. **Do the intervals overlap?** Five point estimates always look like a trend.
   Whether that trend is real is a different question.

In [ ]:
import pandas as pd

from qwen_text2sql.reporting import (
    artifact,
    environment_report,
    load_records,
    preflight,
    project_root,
)
from qwen_text2sql.reporting.style import use_house_style

ROOT = project_root()
PALETTE = use_house_style("light")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
ROOT

In [ ]:
report = environment_report()
print(f"commit   {report['git_commit']}")
print(f"python   {report['python']}")
print(f"platform {report['platform']}")
pd.Series(report["packages"], name="version").to_frame()

In [ ]:
from dataclasses import asdict

from qwen_text2sql.experiments import learning_curve_plan

pd.DataFrame([asdict(cell) for cell in learning_curve_plan(6600)])

Run the sweep with:

```bash
poetry run python scripts/run_sweep.py --kind learning_curve \
  --config configs/qwen35_4b_qlora.yaml \
  --train-data data/processed/bird_train.jsonl \
  --validation-data data/processed/bird_validation.jsonl
```

Every cell is trained from the same seed on a nested subset, so the curve
reflects data quantity rather than which examples happened to be drawn.

In [ ]:
pd.DataFrame(preflight("learning_curve_summary", "baseline_records"))

In [ ]:
summary = pd.read_csv(artifact("learning_curve_summary").require())
summary

## Adding uncertainty

If the sweep recorded per-example records, each point gets a Wilson interval. A
curve of bare point estimates invites the reader to see a trend that the data may
not support.

In [ ]:
from qwen_text2sql.evaluation.intervals import wilson_interval

if {"lower", "upper"}.issubset(summary.columns):
    curve = summary.copy()
elif {"execution_accuracy", "n"}.issubset(summary.columns):
    intervals = [
        wilson_interval(round(row.execution_accuracy * row.n), int(row.n))
        for row in summary.itertuples(index=False)
    ]
    curve = summary.assign(
        lower=[interval.lower for interval in intervals],
        upper=[interval.upper for interval in intervals],
    )
else:
    curve = summary.copy()
    print(
        "No per-cell sample size in the summary, so no interval can be drawn.\n"
        "The curve below shows point estimates only — read it accordingly."
    )
curve

In [ ]:
baseline_accuracy = None
if artifact("baseline_records").exists:
    baseline_accuracy = float(
        load_records(artifact("baseline_records").path)["execution_match"].mean()
    )
    print(f"Unadapted baseline: {baseline_accuracy:.1%}")
else:
    print("No baseline records; the curve will be drawn without a reference line.")

In [ ]:
from qwen_text2sql.reporting import figures

figures.learning_curve(curve, baseline=baseline_accuracy)

## Where does it cross, and where does it flatten?

In [ ]:
ordered = curve.sort_values("train_size")

if baseline_accuracy is not None:
    above = ordered[ordered["execution_accuracy"] > baseline_accuracy]
    if above.empty:
        print("The curve never exceeds the unadapted baseline at these sizes.")
    else:
        print(f"First size beating the baseline: {int(above.iloc[0]['train_size']):,} examples")

deltas = ordered["execution_accuracy"].diff()
marginal = pd.DataFrame(
    {
        "train_size": ordered["train_size"],
        "execution_accuracy": ordered["execution_accuracy"],
        "gain_over_previous": deltas,
        "gain_per_1k_added": deltas / ordered["train_size"].diff() * 1000,
    }
)
marginal.style.format({"execution_accuracy": "{:.1%}", "gain_over_previous": "{:+.1%}"}, na_rep="—")

`gain_per_1k_added` is the number to act on: it is the return on the next
thousand annotated examples. When it approaches zero, the answer to "should we
label more data?" is no, and the next experiment should target the failure modes
in [`06_error_analysis`](06_error_analysis.ipynb) instead.

## Reading this curve

**What it shows.** How execution accuracy responds to training-set size for this
model, recipe and evaluation set.

**What it does not show.**

- **A law.** This is one seed per point. Two seeds at the same size can differ by
  more than two adjacent points do, so a non-monotone kink is usually noise.
- **Where the ceiling is.** Flattening here means *this recipe* saturates, not
  that the task does.
- **Independent points.** Nested subsets share examples, so adjacent points are
  correlated and the curve is smoother than independent runs would be.

---

### Provenance

Every number above is produced by code in `src/qwen_text2sql/`, which is unit
tested and type checked. This notebook orchestrates and interprets; it does not
define the analysis. If a figure here looks wrong, the fix belongs in `src/` and
comes with a test.